In [1]:
# Cell 1: 安装依赖
# 只有第一次运行需要跑，跑完看到 Successfully installed 即可
!pip install librosa tqdm
print("✅ 依赖库安装完成！")

Looking in indexes: http://pip.modelarts.private.com:8888/repository/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 18.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.9 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 23.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.9/208.9 kB 26.1 MB/s eta 0:00:00
✅ 依赖库安装完成！


In [3]:
# Cell 2: 数据搬运
import moxing as mox
import os
import time

# ================= ⚠️ 修改这里 ⚠️ =================
# 把下面的 'bullying2' 换成你真实的桶名
OBS_PATH = "obs://bullying2/raw_data"
# ==================================================

LOCAL_PATH = "/cache/dataset"

if not os.path.exists(LOCAL_PATH):
    os.makedirs(LOCAL_PATH)

print(f"🚀 正在从 {OBS_PATH} 下载数据...")
start = time.time()

try:
    # 递归复制整个文件夹，速度极快
    mox.file.copy_parallel(OBS_PATH, LOCAL_PATH)
    print(f"✅ 下载完成！耗时: {time.time() - start:.2f} 秒")
    
    # 检查一下文件还在不在
    print("📂 本地目录检查:", os.listdir(LOCAL_PATH))
    # 应该能看到 ['S01', 'MIVIA_DB4_dist']
except Exception as e:
    print(f"❌ 下载失败: {e}")
    print("👉 请检查 OBS_PATH 里的桶名是否填对？OBS 里有没有 raw_data 文件夹？")

INFO:root:Multiprocessing connection patch for bpo-17560 not applied, not an applicable Python version: 3.11.10 (main, Oct  3 2024, 07:21:51) [GCC 11.2.0]


🚀 正在从 obs://bullying2/raw_data 下载数据...


INFO:root:Listing OBS: 1000
INFO:root:Listing OBS: 2000
INFO:root:Listing OBS: 3000
INFO:root:Listing OBS: 4000
INFO:root:Listing OBS: 5000
INFO:root:Listing OBS: 6000
INFO:root:Listing OBS: 7000
INFO:root:Listing OBS: 8000
INFO:root:Listing OBS: 9000
INFO:root:Listing OBS: 10000
INFO:root:Listing OBS: 11000
INFO:root:Listing OBS: 12000
INFO:root:Listing OBS: 13000
INFO:root:Listing OBS: 14000
INFO:root:Listing OBS: 15000
INFO:root:Listing OBS: 16000
INFO:root:Listing OBS: 17000
INFO:root:Listing OBS: 18000
INFO:root:Listing OBS: 19000
INFO:root:Listing OBS: 20000
INFO:root:Listing OBS: 21000
INFO:root:Listing OBS: 22000
INFO:root:Listing OBS: 23000
INFO:root:Listing OBS: 24000
INFO:root:Listing OBS: 25000
INFO:root:Listing OBS: 26000
INFO:root:Listing OBS: 27000
INFO:root:Listing OBS: 28000
INFO:root:Listing OBS: 29000
INFO:root:Listing OBS: 30000
INFO:root:Listing OBS: 31000
INFO:root:Listing OBS: 32000
INFO:root:Listing OBS: 33000
INFO:root:Listing OBS: 34000
INFO:root:Listing OBS: 

✅ 下载完成！耗时: 61.82 秒
📂 本地目录检查: ['MIVIA_DB4_dist', 'S01']


In [4]:
# Cell 3: 数据清洗与生成
import os
import glob
import numpy as np
import librosa
import xml.etree.ElementTree as ET
from tqdm import tqdm

# ================= 配置路径 =================
DATA_ROOT = "/cache/dataset"
OUTPUT_DIR = "/home/ma-user/work/Ascend_Processed_Data"  # 结果存这里，重启不丢失

# 自动适配你的目录结构
MIVIA_ROOT = os.path.join(DATA_ROOT, "MIVIA_DB4_dist")
XML_DIR = os.path.join(MIVIA_ROOT, "training")
WAV_DIR = os.path.join(MIVIA_ROOT, "training", "sounds")
RADAR_ROOT = os.path.join(DATA_ROOT, "S01")

# 参数设置
AUDIO_SR = 16000     # 16k采样率
MAX_POINTS = 64      # 雷达点数
# 动作映射: A20-A23为异常(1)，A01-A04为正常(0)
ACTION_MAP = {'A22': 1, 'A23': 1, 'A20': 1, 'A21': 1, 'A01': 0, 'A02': 0, 'A03': 0, 'A04': 0}

# ================= 核心函数 =================
def find_wav(xml_name):
    """自动匹配 00001.wav 或 00001_00.wav"""
    base = os.path.splitext(xml_name)[0]
    p1 = os.path.join(WAV_DIR, base + ".wav")
    if os.path.exists(p1): return p1
    p2 = os.path.join(WAV_DIR, base + "_00.wav") # 适配你的文件后缀
    if os.path.exists(p2): return p2
    return None

def read_bin(path):
    try:
        raw = np.fromfile(path, dtype=np.float32)
        if raw.size % 5 != 0: return None
        return raw.reshape(-1, 5)
    except: return None

# ================= 主程序 =================
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

print("🚀 开始数据清洗...")

# --- 1. 处理音频 ---
xml_files = glob.glob(os.path.join(XML_DIR, "*.xml"))
audio_X, audio_y = [], []

print(f"🎤 扫描音频 XML: 找到 {len(xml_files)} 个文件")
for xml in tqdm(xml_files, desc="Processing Audio"):
    wav_path = find_wav(os.path.basename(xml))
    if not wav_path: continue
    
    try:
        tree = ET.parse(xml)
        root = tree.getroot()
        times = []
        # 提取尖叫时间段
        for item in root.findall(".//item"):
            cls = item.find("CLASS_NAME")
            if cls is not None and "scream" in cls.text.lower():
                times.append((float(item.find("STARTSECOND").text), float(item.find("ENDSECOND").text)))
        
        if times:
            # 只有当有尖叫时才加载音频
            y, _ = librosa.load(wav_path, sr=AUDIO_SR)
            for s, e in times:
                # 切片 & 填充到 1秒 (16000点)
                clip = y[int(s*AUDIO_SR):int(e*AUDIO_SR)]
                if len(clip) > 16000: clip = clip[:16000]
                else: clip = np.pad(clip, (0, 16000-len(clip)), 'constant')
                audio_X.append(clip)
                audio_y.append(1) # 标签 1 = 尖叫
    except: continue

if audio_X:
    np.save(os.path.join(OUTPUT_DIR, "audio_X.npy"), np.stack(audio_X).astype(np.float32))
    np.save(os.path.join(OUTPUT_DIR, "audio_y.npy"), np.array(audio_y, dtype=np.int32))
    print(f"✅ 音频处理完毕: 提取了 {len(audio_X)} 个尖叫样本")
else:
    print("⚠️ 警告: 未提取到音频样本，请检查 OBS 路径结构！")

# --- 2. 处理雷达 ---
radar_X, radar_y = [], []
print(f"📡 扫描雷达数据...")

for act, lbl in ACTION_MAP.items():
    files = glob.glob(os.path.join(RADAR_ROOT, "**", act, "mmwave", "*.bin"), recursive=True)
    print(f"   -> 动作 {act}: 找到 {len(files)} 帧")
    
    for f in files:
        pc = read_bin(f)
        if pc is None: continue
        
        # 采样/填充到 64点
        n = pc.shape[0]
        if n >= MAX_POINTS:
            pc = pc[np.random.choice(n, MAX_POINTS, replace=False), :]
        else:
            pc = np.vstack((pc, np.zeros((MAX_POINTS-n, 5))))
            
        radar_X.append(pc)
        radar_y.append(lbl)

if radar_X:
    np.save(os.path.join(OUTPUT_DIR, "radar_X.npy"), np.stack(radar_X).astype(np.float32))
    np.save(os.path.join(OUTPUT_DIR, "radar_y.npy"), np.array(radar_y, dtype=np.int32))
    print(f"✅ 雷达处理完毕: 提取了 {len(radar_X)} 个样本")
else:
    print("⚠️ 警告: 未提取到雷达数据！")

print(f"\n🎉 所有数据已保存至: {OUTPUT_DIR}")
print("下一步：编写 MindSpore 训练代码读取这些 .npy 文件！")

🚀 开始数据清洗...
🎤 扫描音频 XML: 找到 66 个文件


Processing Audio:   0%|          | 0/66 [00:00<?, ?it/s]/home/ma-user/anaconda3/envs/MindSpore/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing Audio: 100%|██████████| 66/66 [00:27<00:00,  2.37it/s]


✅ 音频处理完毕: 提取了 700 个尖叫样本
📡 扫描雷达数据...
   -> 动作 A22: 找到 297 帧
   -> 动作 A23: 找到 297 帧
   -> 动作 A20: 找到 297 帧
   -> 动作 A21: 找到 297 帧
   -> 动作 A01: 找到 297 帧
   -> 动作 A02: 找到 297 帧
   -> 动作 A03: 找到 297 帧
   -> 动作 A04: 找到 297 帧
✅ 雷达处理完毕: 提取了 2376 个样本

🎉 所有数据已保存至: /home/ma-user/work/Ascend_Processed_Data
下一步：编写 MindSpore 训练代码读取这些 .npy 文件！


In [2]:
# Cell 4: 模型定义与训练
import os
import numpy as np
import mindspore
import mindspore.nn as nn
import mindspore.ops as ops
from mindspore import context, Tensor, Model
from mindspore.train.callback import LossMonitor, TimeMonitor
from mindspore.dataset import GeneratorDataset

context.set_context(mode=context.GRAPH_MODE, device_target="Ascend")

# 1. 数据集定义
class MultiModalDataset:
    def __init__(self, data_dir):
        self.radar_X = np.load(os.path.join(data_dir, "radar_X.npy"))
        self.radar_y = np.load(os.path.join(data_dir, "radar_y.npy"))
        self.audio_X = np.load(os.path.join(data_dir, "audio_X.npy"))
        self.pos_indices = np.where(self.radar_y == 1)[0]
    
    def __getitem__(self, index):
        radar_sample = self.radar_X[index]
        label = self.radar_y[index]
        radar_tensor = radar_sample.transpose(1, 0).astype(np.float32)
        
        if label == 1:
            rand_idx = np.random.randint(0, len(self.audio_X))
            audio_sample = self.audio_X[rand_idx]
        else:
            audio_sample = np.random.randn(16000) * 0.01
        audio_tensor = audio_sample.reshape(1, 16000).astype(np.float32)
        
        return radar_tensor, audio_tensor, np.array(label, dtype=np.int32)

    def __len__(self):
        return len(self.radar_X)

# 2. 网络定义
class AscendSentinelNet(nn.Cell):
    def __init__(self):
        super(AscendSentinelNet, self).__init__()
        self.radar_branch = nn.SequentialCell([
            nn.Conv1d(5, 32, kernel_size=3, pad_mode='same'),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, pad_mode='same'),
            nn.ReLU(), nn.AdaptiveAvgPool1d(1), nn.Flatten()
        ])
        self.audio_branch = nn.SequentialCell([
            nn.Conv1d(1, 16, kernel_size=64, stride=16, pad_mode='valid'),
            nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(16, 32, kernel_size=3, pad_mode='same'),
            nn.ReLU(), nn.AdaptiveAvgPool1d(1), nn.Flatten()
        ])
        self.classifier = nn.SequentialCell([
            nn.Dense(64 + 32, 64), nn.ReLU(), nn.Dense(64, 2)
        ])

    def construct(self, r, a):
        feat_r = self.radar_branch(r)
        feat_a = self.audio_branch(a)
        combined = ops.concat((feat_r, feat_a), axis=1)
        return self.classifier(combined)

# 3. 自定义训练连接层 (Fix for Loss Error)
class CustomWithLossCell(nn.Cell):
    def __init__(self, backbone, loss_fn):
        super(CustomWithLossCell, self).__init__(auto_prefix=False)
        self._backbone = backbone
        self._loss_fn = loss_fn
    def construct(self, radar, audio, label):
        logits = self._backbone(radar, audio)
        return self._loss_fn(logits, label)

# 4. 训练流程
data_dir = "/home/ma-user/work/Ascend_Processed_Data"
ds = GeneratorDataset(MultiModalDataset(data_dir), ["radar", "audio", "label"], shuffle=True)
ds = ds.batch(32)

net = AscendSentinelNet()
loss_fn = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction='mean')
optimizer = nn.Adam(net.trainable_params(), learning_rate=0.001)

# 封装网络
net_with_loss = CustomWithLossCell(net, loss_fn)
model = Model(net_with_loss, optimizer=optimizer)

print("🚀 开始训练...")
model.train(epoch=10, train_dataset=ds, callbacks=[LossMonitor(per_print_times=20), TimeMonitor()])

print("✅ 训练完成")
mindspore.save_checkpoint(net, "/home/ma-user/work/sentinel_model.ckpt")

[WARNING] ME(2498:281473296748768,MainProcess):2026-03-17-16:22:40.937.000 [mindspore/run_check/_check_version.py:409] Can not find the tbe operator implementation(need by mindspore-ascend). Please check whether the Environment Variable PYTHONPATH is set. For details, refer to the installation guidelines: https://www.mindspore.cn/install
[WARNING] ME(2498:281473296748768,MainProcess):2026-03-17-16:22:40.939.000 [mindspore/context.py:1418] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.
[WARNING] ME(2498:281473296748768,MainProcess):2026-03-17-16:22:40.940.000 [mindspore/run_check/_check_version.py:409] Can not find the tbe operator implementation(need by mindspore-ascend). Please check whether the Environment Variable PYTHONPATH is set. For details, refer to the installation guidelines: https://www.mindspore.cn/install


🚀 开始训练...


[WARNING] CORE(2498,ffff9bddd0e0,python):2026-03-17-16:22:41.416.343 [mindspore/core/utils/ms_context.cc:537] GetJitLevel] Set jit level to O2 for rank table startup method.
[ERROR] CORE(2498,ffff9bddd0e0,python):2026-03-17-16:22:42.520.964 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_2498/2598085750.py]
[WARNING] CORE(2498,ffff9bddd0e0,python):2026-03-17-16:22:42.521.010 [mindspore/core/utils/info.cc:94] ReadSectionDebugInfoFromFile] The file '/tmp/ipykernel_2498/2598085750.py' may not exists.


...epoch: 1 step: 20, loss is 0.38573914766311646
epoch: 1 step: 40, loss is 0.30070579051971436
epoch: 1 step: 60, loss is 0.3130255341529846


[ERROR] CORE(2498,ffff9bddd0e0,python):2026-03-17-16:23:47.281.023 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_2498/2598085750.py]
[WARNING] CORE(2498,ffff9bddd0e0,python):2026-03-17-16:23:47.281.086 [mindspore/core/utils/info.cc:94] ReadSectionDebugInfoFromFile] The file '/tmp/ipykernel_2498/2598085750.py' may not exists.


..Train epoch time: 80812.124 ms, per step time: 1077.495 ms
epoch: 2 step: 5, loss is 0.2865520119667053
epoch: 2 step: 25, loss is 0.17106172442436218
epoch: 2 step: 45, loss is 0.11704201996326447
epoch: 2 step: 65, loss is 0.06849707663059235
Train epoch time: 7264.653 ms, per step time: 96.862 ms
epoch: 3 step: 10, loss is 0.028301598504185677
epoch: 3 step: 30, loss is 0.013339770026504993
epoch: 3 step: 50, loss is 0.006107506342232227
epoch: 3 step: 70, loss is 0.0037642531096935272
Train epoch time: 7253.636 ms, per step time: 96.715 ms
epoch: 4 step: 15, loss is 0.001619769842363894
epoch: 4 step: 35, loss is 0.0009417408145964146
epoch: 4 step: 55, loss is 0.0025683448184281588
epoch: 4 step: 75, loss is 0.00121603743173182
Train epoch time: 7255.569 ms, per step time: 96.741 ms
epoch: 5 step: 20, loss is 0.0010719203855842352
epoch: 5 step: 40, loss is 0.001077119610272348
epoch: 5 step: 60, loss is 0.0020622943993657827
Train epoch time: 7242.663 ms, per step time: 96.569 

In [1]:
import numpy as np
import mindspore as ms
from mindspore import Tensor, export

net = AscendSentinelNet()
ms.load_param_into_net(net, ms.load_checkpoint("/home/ma-user/work/sentinel_model.ckpt"))
net.set_train(False)

# 🔥 终极绝杀：开启 O2 级自动混合精度！网络会自动分配 FP16 和 FP32
net = ms.amp.auto_mixed_precision(net, amp_level="O2")

# 输入数据恢复成正常的 FP32，O2 网络会在内部自动进行无缝转换
input_radar = Tensor(np.zeros([1, 5, 64]), ms.float32)
input_audio = Tensor(np.zeros([1, 1, 16000]), ms.float32)

output_path = "/home/ma-user/work/sentinel"
print(f"🚀 导出 O2 混合精度模型至: {output_path}.mindir")
export(net, input_radar, input_audio, file_name=output_path, file_format="MINDIR")
print("✅ 导出成功！")

[WARNING] ME(2498:281473296748768,MainProcess):2026-03-17-16:21:19.573.000 [mindspore/run_check/_check_version.py:409] Can not find the tbe operator implementation(need by mindspore-ascend). Please check whether the Environment Variable PYTHONPATH is set. For details, refer to the installation guidelines: https://www.mindspore.cn/install
/home/ma-user/anaconda3/envs/MindSpore/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/ma-user/anaconda3/envs/MindSpore/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/home/ma-user/anaconda3/envs/MindSpore/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type 

NameError: name 'AscendSentinelNet' is not defined

In [12]:
# Cell 6: 现场盲测验证
import numpy as np
import mindspore
from mindspore import Tensor

# 准备
data_dir = "/home/ma-user/work/Ascend_Processed_Data"
# 这里需要重新实例化一次 Dataset 类以便访问数据，虽然有点冗余但能保证独立运行
dataset_check = MultiModalDataset(data_dir)

net = AscendSentinelNet()
mindspore.load_param_into_net(net, mindspore.load_checkpoint("/home/ma-user/work/sentinel_model.ckpt"))

# 🔥 关键修复
net.set_train(False)

def predict_single(index):
    radar_sample = dataset_check.radar_X[index]
    label_true = dataset_check.radar_y[index]
    
    if label_true == 1:
        audio_idx = np.random.randint(0, len(dataset_check.audio_X))
        audio_sample = dataset_check.audio_X[audio_idx]
    else:
        audio_sample = np.random.randn(16000) * 0.01

    r_tensor = Tensor(radar_sample.transpose(1, 0).reshape(1, 5, 64), mindspore.float32)
    a_tensor = Tensor(audio_sample.reshape(1, 1, 16000), mindspore.float32)

    logits = net(r_tensor, a_tensor)
    probs = mindspore.ops.Softmax(axis=1)(logits).asnumpy()[0]
    pred_label = np.argmax(probs)
    
    status = "✅" if pred_label == label_true else "❌"
    print(f"ID: {index:<5} | 真值: {label_true} -> 预测: {pred_label} | 正常:{probs[0]:.2f}/异常:{probs[1]:.2f} | {status}")

print("🔍 --- 验证开始 ---")
print("正样本抽查 (应为 1):")
for i in np.random.choice(np.where(dataset_check.radar_y == 1)[0], 5): predict_single(i)
print("\n负样本抽查 (应为 0):")
for i in np.random.choice(np.where(dataset_check.radar_y == 0)[0], 5): predict_single(i)

🔍 --- 验证开始 ---
正样本抽查 (应为 1):


[ERROR] CORE(2691,ffffa6ffe0e0,python):2026-02-05-00:17:14.929.539 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_2691/2598085750.py]
[WARNING] CORE(2691,ffffa6ffe0e0,python):2026-02-05-00:17:14.929.594 [mindspore/core/utils/info.cc:94] ReadSectionDebugInfoFromFile] The file '/tmp/ipykernel_2691/2598085750.py' may not exists.
[ERROR] CORE(2691,ffffa6ffe0e0,python):2026-02-05-00:17:14.929.901 [mindspore/core/utils/file_utils.cc:253] GetRealPath] Get realpath failed, path[/tmp/ipykernel_2691/2598085750.py]
[WARNING] CORE(2691,ffffa6ffe0e0,python):2026-02-05-00:17:14.929.927 [mindspore/core/utils/info.cc:94] ReadSectionDebugInfoFromFile] The file '/tmp/ipykernel_2691/2598085750.py' may not exists.


.ID: 1105  | 真值: 1 -> 预测: 1 | 正常:0.00/异常:1.00 | ✅
ID: 88    | 真值: 1 -> 预测: 1 | 正常:0.00/异常:1.00 | ✅
ID: 18    | 真值: 1 -> 预测: 1 | 正常:0.00/异常:1.00 | ✅
ID: 690   | 真值: 1 -> 预测: 1 | 正常:0.00/异常:1.00 | ✅
ID: 317   | 真值: 1 -> 预测: 1 | 正常:0.00/异常:1.00 | ✅

负样本抽查 (应为 0):
ID: 1342  | 真值: 0 -> 预测: 0 | 正常:1.00/异常:0.00 | ✅
ID: 1795  | 真值: 0 -> 预测: 0 | 正常:1.00/异常:0.00 | ✅
ID: 1646  | 真值: 0 -> 预测: 0 | 正常:1.00/异常:0.00 | ✅
ID: 1728  | 真值: 0 -> 预测: 0 | 正常:1.00/异常:0.00 | ✅
ID: 2089  | 真值: 0 -> 预测: 0 | 正常:1.00/异常:0.00 | ✅


In [ ]:
import numpy as np
import mindspore as ms

# 1. 重新实例化一个全新、干净的 FP32 网络本体！不被之前的操作污染！
clean_net = AscendSentinelNet()
ms.load_param_into_net(clean_net, ms.load_checkpoint("sentinel_model.ckpt")) # 确保路径正确
clean_net.set_train(False)
clean_net.to_float(ms.float32) # 强行确保是最高精度的 FP32

# 2. 找到所有正常样本的索引
normal_indices = np.where(dataset_check.radar_y == 0)[0]

valid_idx = -1
for idx in normal_indices:
    radar_sample = dataset_check.radar_X[idx]
    
    if np.isnan(radar_sample).any():
        continue 
        
    real_radar = radar_sample.transpose(1, 0).reshape(1, 5, 64).astype(np.float32)
    real_audio = (np.random.randn(1, 1, 16000) * 0.01).astype(np.float32)
    
    # 🌟 核心：用干净的 FP32 网络做前向推理，绝对不会再爆 NaN！
    logits_cloud = clean_net(ms.Tensor(real_radar), ms.Tensor(real_audio))
    
    if not np.isnan(logits_cloud.asnumpy()).any():
        valid_idx = idx
        break

if valid_idx != -1:
    print(f"✅ 成功找到健康的【正常/不霸凌】样本！索引是: {valid_idx}")
    np.save("real_radar_normal.npy", real_radar)
    np.save("real_audio_normal.npy", real_audio)
    
    probs = ms.ops.Softmax(axis=1)(logits_cloud).asnumpy()[0]
    print(f"☁️ 云端 Logits 输出: {logits_cloud.asnumpy()}")
    print(f"☁️ 云端预测概率 -> 正常: {probs[0]*100:.2f}%, 霸凌: {probs[1]*100:.2f}%")
else:
    print("❌ 还是不行，请检查雷达数据本身是否损坏。")